In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy
import copy

from scipy.sparse import coo_matrix, block_diag, identity, hstack, csr_matrix, csc_matrix, vstack
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time 

from pyiga import assemble, bspline, vform, geometry, vis, solvers, utils, topology, ieti, algebra, operators, adaptive
from pyiga import algebra_cy, ieti_cy, bspline_cy

from scipy.sparse.linalg import aslinearoperator as LinOp

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)

In [2]:
def Fichera(deg,N):
    kvs = 7*(3 * (bspline.make_knots(deg, 0.0, 1.0, N),),)
    
    #define geometry
    geos = [
        geometry.unit_cube(),
        geometry.unit_cube().translate((0,0,-1)),
        geometry.unit_cube().translate((0,-1,0)),
        geometry.unit_cube().translate((-1,0,0)),
        geometry.unit_cube().translate((-1,-1,0)),
        geometry.unit_cube().translate((-1,0,-1)),
        geometry.unit_cube().translate((0,-1,-1)),
        geometry.unit_cube().translate((-1,-1,-1))
    ]
    
    patches = [(k, g) for k, g in zip(kvs,geos)]
    M = topology.MultiPatch3D(patches)
    return M

In [7]:
alpha = 2
u = lambda x,y,z: np.sin(2*np.pi*x)*np.sin(2*np.pi*y)*np.sin(2*np.pi*z)
f = lambda x,y,z: 12*np.pi**2*np.sin(2*np.pi*x)*np.sin(2*np.pi*y)*np.sin(2*np.pi*z)

M = Fichera(3,2)

In [11]:
u = lambda x,y,z: np.sqrt(x**2+y**2+z**2)
f = lambda x,y,z: -2/np.sqrt(x**2+y**2+z**2)

In [14]:
M = Fichera(3,2)
for i in range(1):
    M.h_refine(-1)
    MB = assemble.MultiBasis(M, subspace='C0')
    Mh = MB.assemble_volume('u * v *dx', arity=2)
    Kh = MB.assemble_volume('inner(grad(u),grad(v))*dx', arity=2)
    Fh = MB.assemble_volume('f * v * dx', arity=1, f=f)
    Uh = MB.assemble_volume('f * v * dx', arity=1, f=u)
    u_exact = operators.make_solver(Mh, spd=False)@Uh

    dir_bcs = MB.set_fixed_boundary({0:u})
    LS = assemble.RestrictedLinearSystem(Kh,Fh,dir_bcs)
    uh = LS.complete(operators.make_solver(LS.A, spd=False)@LS.b)
    eh=uh-u_exact
    print(np.sqrt(eh@Kh@eh))

setting up constraints took 0.040886878967285156 seconds.
Basis setup took 0.0026786327362060547 seconds
3.958529804919748


In [17]:
clear(Mh)

In [82]:
MB = assemble.MultiBasis(M, subspace='C0')

setting up constraints took 0.08449077606201172 seconds.
Basis setup took 0.015715837478637695 seconds


In [83]:
Mh = MB.assemble_volume('u * v *dx', arity=2)
Kh = MB.assemble_volume('inner(grad(u),grad(v))*dx', arity=2)
Fh = MB.assemble_volume('f * v * dx', arity=1, f=f)
Uh = MB.assemble_volume('f * v * dx', arity=1, f=u)
u_exact = operators.make_solver(Mh)@Uh

In [84]:
dir_bcs = MB.set_fixed_boundary({0:0})

In [85]:
LS = assemble.RestrictedLinearSystem(Kh,Fh,dir_bcs)

In [86]:
uh = LS.complete(operators.make_solver(LS.A, spd=True)@LS.b)

In [87]:
eh=uh-u_exact

In [89]:
np.sqrt(eh@Mh@eh)

np.float64(0.229048584915279)